# Reddit GLP-1 RA posts: data preparation, LLM extraction and analysis-field derivation

Code accompanying *"Cost Strategies and Regimen Changes Discussed in Online GLP-1 Receptor Agonist Communities: Large Language Model-Assisted Analysis of Reddit Posts"*.

The notebook runs the pipeline end to end:

| Step | What it does | Output |
|---|---|---|
| 1 | Convert raw Arctic Shift `.jsonl` files to a fixed set of fields | `cleaned/*.jsonl` |
| 2 | Remove deleted/removed posts, null records, duplicate post IDs, deleted authors and empty bodies; build `combined_text` | same files |
| 3 | Normalise brand names and misspellings to generic drug names | same files, `weight_loss_drug_list.csv` |
| 4 | Keep posts that mention at least one weight-loss drug | same files |
| 5 | Combine files, add emoji columns, restrict dates, sort newest first, assign batches | `llm/combined_posts.jsonl`, `llm/batch_mapping.csv` |
| 6 | Extract 21 fields per post with Gemini 2.5 Flash-Lite (temperature 0.0) | `llm/gemini_results_batch_N.csv` |
| 7 | Concatenate batch outputs | `llm/gemini_results_combined.csv` |
| 8 | Build the analytic corpus (exclusions, de-duplication, molecule assignment) and derive the analysis fields | `analytic_corpus.csv` |

**Data.** Post text is not distributed with this code (see the Data Availability statement). Obtain posts from the Arctic Shift archive and place the per-subreddit `.jsonl` files in `data/raw_posts/`.

**Credentials.** Step 6 reads the Gemini key from the environment variable `GEMINI_API_KEY`. Never write a key into the notebook.

**Coverage.** Batches are numbered newest first (batch 1 = most recent). Batches 1-20, which cover 6 August 2023 to 31 December 2025, were run for this study; batches 21-25 (1 January 2021 to 6 August 2023) were prepared but not run.

**Requirements.** Python 3.10+; `pip install pandas emoji google-genai pydantic tenacity tqdm`.

In [ ]:
import asyncio
import csv
import json
import math
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import pandas as pd

# All paths are relative to DATA_DIR (override with the GLP1_DATA_DIR environment variable).
DATA_DIR = Path(os.environ.get("GLP1_DATA_DIR", "data"))
RAW_DIR = DATA_DIR / "raw_posts"      # Arctic Shift .jsonl files, one per subreddit
CLEAN_DIR = DATA_DIR / "cleaned"
LLM_DIR = DATA_DIR / "llm"
for d in (CLEAN_DIR, LLM_DIR):
    d.mkdir(parents=True, exist_ok=True)

COMBINED_FILE = LLM_DIR / "combined_posts.jsonl"
BATCH_MAP_FILE = LLM_DIR / "batch_mapping.csv"
RESULTS_FILE = LLM_DIR / "gemini_results_combined.csv"

DATE_FROM, DATE_TO = "2021-01-01", "2026-05-14"   # extraction candidate window (Step 5)
ANALYSIS_FROM, ANALYSIS_TO = "2021-01-01", "2025-12-31"  # window applied in Step 8
BATCH_SIZE = 9900
MODEL = "gemini-2.5-flash-lite"
TEMPERATURE = 0.0
CONCURRENCY = 12

## Step 1. Raw files to cleaned records

In [ ]:
RAW_FIELDS = [
    "id", "created_utc", "author", "author_fullname", "subreddit",
    "subreddit_subscribers", "url", "ups", "upvote_ratio",
    "num_comments", "score", "permalink", "author_flair_text",
    "link_flair_text", "selftext", "title",
]


def raw_to_clean(record: dict, source_file: str) -> dict:
    cleaned = {"reddit_post_id": record.get("id")}
    for field in RAW_FIELDS:
        cleaned[field] = record.get(field)
    cleaned["source_file"] = source_file
    cleaned["created_dt"] = cleaned["created_date"] = None
    utc_ts = record.get("created_utc")
    if utc_ts:
        try:
            dt = datetime.fromtimestamp(float(utc_ts), tz=timezone.utc)
            cleaned["created_dt"] = dt.strftime("%Y-%m-%d %H:%M:%S")
            cleaned["created_date"] = dt.strftime("%Y-%m-%d")
        except (ValueError, OSError):
            pass
    return cleaned


def read_jsonl(path: Path) -> list:
    records = []
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    continue
    return records


def write_jsonl(path: Path, records: list) -> None:
    with open(path, "w", encoding="utf-8") as fh:
        for rec in records:
            fh.write(json.dumps(rec) + "\n")


raw_files = sorted(RAW_DIR.rglob("*.jsonl"))
print(f"Found {len(raw_files)} raw JSONL files")

## Steps 2-4. Cleaning, drug-name normalisation and drug filter

Each file is processed in memory and written once. The order of operations is: remove deleted or removed posts and null records, remove duplicate post IDs, remove deleted authors and empty bodies, build `combined_text` (title + body), normalise drug names, then keep posts that mention a weight-loss drug.

In [ ]:
# --- Weight Loss Drug Name Normalization Mappings ---

# Brand / commercial name to RxNorm standard generic mapping
brand_to_generic = {
    # ── GLP-1 / GIP Receptor Agonists — Semaglutide ──────────────────
    'ozempic': 'semaglutide',           # SC injection, T2D
    'wegovy': 'semaglutide',            # SC injection, obesity
    'rybelsus': 'semaglutide',          # oral tablet
    # ── GLP-1 / GIP Dual Agonist — Tirzepatide ───────────────────────
    'mounjaro': 'tirzepatide',          # SC injection, T2D
    'zepbound': 'tirzepatide',          # SC injection, obesity
    # ── GLP-1 Receptor Agonist — Liraglutide ─────────────────────────
    'victoza': 'liraglutide',           # SC injection, T2D
    'saxenda': 'liraglutide',           # SC injection, obesity
    # ── GLP-1 Receptor Agonist — Dulaglutide ─────────────────────────
    'trulicity': 'dulaglutide',
    # ── GLP-1 Receptor Agonist — Exenatide ───────────────────────────
    'byetta': 'exenatide',
    'bydureon': 'exenatide',
    'bydureon bcise': 'exenatide',
    # ── GLP-1 Receptor Agonist — Albiglutide ─────────────────────────
    'tanzeum': 'albiglutide',
    # ── GLP-1 Receptor Agonist — Lixisenatide ────────────────────────
    'adlyxin': 'lixisenatide',
    'lyxumia': 'lixisenatide',
    # ── GLP-1 Receptor Agonist — Semaglutide oral (brand alt) ────────
    'ozempic oral': 'semaglutide',
    # ── Phentermine / Sympathomimetics ───────────────────────────────
    'adipex': 'phentermine',
    'adipex-p': 'phentermine',
    'lomaira': 'phentermine',
    'fastin': 'phentermine',
    'ionamin': 'phentermine',
    'suprenza': 'phentermine',
    'pro-fast': 'phentermine',
    # ── Combination Weight Loss Medications ──────────────────────────
    'qsymia': 'phentermine/topiramate',
    'contrave': 'naltrexone/bupropion',
    # ── Lipase Inhibitor ──────────────────────────────────────────────
    'xenical': 'orlistat',
    'alli': 'orlistat',
    # ── Serotonin Agonist (withdrawn 2020, still discussed) ───────────
    'belviq': 'lorcaserin',
    'belviq xr': 'lorcaserin',
    # ── Other Sympathomimetics ────────────────────────────────────────
    'didrex': 'benzphetamine',
    'tenuate': 'diethylpropion',
    'tenuate dospan': 'diethylpropion',
    'bontril': 'phendimetrazine',
    'plegine': 'phendimetrazine',
    'prelu-2': 'phendimetrazine',
    'melfiat': 'phendimetrazine',
    # ── Topiramate (used off-label / in combination) ──────────────────
    'topamax': 'topiramate',
    'trokendi xr': 'topiramate',
    'qudexy xr': 'topiramate',
    # ── Metformin (used off-label for weight / PCOS) ──────────────────
    'glucophage': 'metformin',
    'glucophage xr': 'metformin',
    'fortamet': 'metformin',
    'glumetza': 'metformin',
    'riomet': 'metformin',
    # ── SGLT2 Inhibitors (sometimes co-prescribed for weight) ─────────
    'jardiance': 'empagliflozin',
    'farxiga': 'dapagliflozin',
    'forxiga': 'dapagliflozin',
    'invokana': 'canagliflozin',
    'steglatro': 'ertugliflozin',
    # ── Naltrexone (standalone) ───────────────────────────────────────
    'vivitrol': 'naltrexone',
    'revia': 'naltrexone',
    # ── Setmelanotide (rare genetic obesity) ─────────────────────────
    'imcivree': 'setmelanotide',
    # ── Cellulose / Hydrogel (non-stimulant) ─────────────────────────
    'plenity': 'cellulose and citric acid hydrogel',
}

# Common misspellings / variants → RxNorm standard generic
misspellings_to_generic = {
    # Semaglutide
    'semoglutide': 'semaglutide',
    'samaglutide': 'semaglutide',
    'semagludite': 'semaglutide',
    'semaglutied': 'semaglutide',
    'semaglutde': 'semaglutide',
    'semaglutdie': 'semaglutide',
    'semaglutíde': 'semaglutide',
    'semiglutide': 'semaglutide',
    # Tirzepatide
    'tirzipatide': 'tirzepatide',
    'tirzepatdie': 'tirzepatide',
    'tizepatide': 'tirzepatide',
    'tirzepetide': 'tirzepatide',
    'tirzepaatide': 'tirzepatide',
    'tirzpatide': 'tirzepatide',
    'terzapatide': 'tirzepatide',
    # Liraglutide
    'liragludite': 'liraglutide',
    'liragulitide': 'liraglutide',
    'liraglutdie': 'liraglutide',
    'liragluitide': 'liraglutide',
    'liragutide': 'liraglutide',
    # Dulaglutide
    'dulaglutdie': 'dulaglutide',
    'dulagluide': 'dulaglutide',
    # Ozempic misspellings (brand → generic)
    'ozempik': 'semaglutide',
    'ozempick': 'semaglutide',
    'ozempec': 'semaglutide',
    'oezmpic': 'semaglutide',
    'ozemipic': 'semaglutide',
    # Wegovy misspellings (brand → generic)
    'wegovi': 'semaglutide',
    'wegovey': 'semaglutide',
    'wgovy': 'semaglutide',
    # Mounjaro misspellings (brand → generic)
    'mountjaro': 'tirzepatide',
    'mounjarro': 'tirzepatide',
    'mountjarro': 'tirzepatide',
    'mounjaroe': 'tirzepatide',
    'moonjaro': 'tirzepatide',
    # Zepbound misspellings (brand → generic)
    'zep bound': 'tirzepatide',
    'zepboud': 'tirzepatide',
    # Phentermine misspellings
    'phentamine': 'phentermine',
    'fentemine': 'phentermine',
    'phentemine': 'phentermine',
    'phentremine': 'phentermine',
    'phenitramine': 'phentermine',
    # Orlistat misspellings
    'orlistate': 'orlistat',
    'orlistaat': 'orlistat',
    'orlistal': 'orlistat',
    # Metformin misspellings
    'metphormin': 'metformin',
    'metfomin': 'metformin',
    'metformine': 'metformin',
    'metphormine': 'metformin',
    # Naltrexone misspellings
    'naltraxone': 'naltrexone',
    'naltroxone': 'naltrexone',
    # Topiramate misspellings
    'topirimate': 'topiramate',
    'topiramite': 'topiramate',
    'topiramat': 'topiramate',
    # Empagliflozin misspellings
    'empaglifozin': 'empagliflozin',
    # Dapagliflozin misspellings
    'dapaglifozin': 'dapagliflozin',
}

# Combine all mappings, sort by length descending so multi-word matches go first
all_mappings = {**brand_to_generic, **misspellings_to_generic}
all_mappings_sorted = sorted(all_mappings.items(), key=lambda x: len(x[0]), reverse=True)

# Pre-compile patterns (longest variant first) and define the normaliser
compiled_patterns = [(re.compile(r"\b" + re.escape(variant) + r"\b", re.IGNORECASE), generic)
                     for variant, generic in all_mappings_sorted]


def normalize_drug_names(text: str) -> str:
    """Replace brand names and misspellings with the generic drug name."""
    for pattern, generic in compiled_patterns:
        text = pattern.sub(generic, text)
    return text


# Terms used to keep posts that mention a weight-loss drug (all variants and generic names)
drug_terms = {t.lower() for pair in all_mappings.items() for t in pair}
drug_patterns = [(term, re.compile(r"\b" + re.escape(term) + r"\b", re.IGNORECASE))
                 for term in sorted(drug_terms, key=len, reverse=True)]


def contains_weight_loss_drug(text: str) -> bool:
    text_lower = text.lower()
    return any(term in text_lower and pattern.search(text) for term, pattern in drug_patterns)


# Save the mapping table for transparency
mapping_rows = (
    [{"variant": v, "generic_name": g, "mapping_type": "brand/commercial"} for v, g in brand_to_generic.items()]
    + [{"variant": v, "generic_name": g, "mapping_type": "misspelling"} for v, g in misspellings_to_generic.items()]
)
(pd.DataFrame(mapping_rows)
   .sort_values(["generic_name", "mapping_type", "variant"])
   .to_csv(CLEAN_DIR / "weight_loss_drug_list.csv", index=False))
print(f"Normalisation mappings: {len(all_mappings)}; match terms: {len(drug_terms)}")

In [ ]:
DELETED_MARKERS = {"[deleted]", "[removed]", ""}


def build_combined_text(r: dict) -> str:
    title = str(r.get("title", "") or "").strip()
    selftext = str(r.get("selftext", "") or "").strip()
    if title and selftext:
        return f"{title}. {selftext}"
    return title or selftext


def clean_records(records: list) -> tuple:
    """Steps 2-4 for one file. Returns (kept_records, audit_counts)."""
    n0 = len(records)
    # Step 2a: drop posts whose body AND title are deleted/removed/empty; drop records with neither field
    records = [r for r in records
               if str(r.get("selftext", "")).strip() not in DELETED_MARKERS
               or str(r.get("title", "")).strip() not in DELETED_MARKERS]
    records = [r for r in records if r.get("title") is not None or r.get("selftext") is not None]
    n1 = len(records)
    # Step 2b: remove duplicate post IDs
    seen, deduped = set(), []
    for r in records:
        pid = r.get("reddit_post_id")
        if pid is None or pid not in seen:
            if pid is not None:
                seen.add(pid)
            deduped.append(r)
    records = deduped
    n2 = len(records)
    # Step 2c: remove deleted authors and empty bodies
    records = [r for r in records if "[deleted]" not in str(r.get("author", ""))]
    records = [r for r in records if r.get("selftext", "") != ""]
    n3 = len(records)
    # Step 2d: build combined_text; Step 3: normalise drug names
    for r in records:
        r["combined_text"] = normalize_drug_names(build_combined_text(r))
    # Step 4: keep posts that mention a weight-loss drug
    records = [r for r in records if contains_weight_loss_drug(r["combined_text"])]
    return records, {"raw_records": n0, "after_deleted_and_null": n1, "after_duplicate_ids": n2,
                     "after_author_and_empty_body": n3, "with_drug_mention": len(records)}


audit_rows = []
for path in raw_files:
    with open(path, "rb") as fh:
        first_byte = fh.read(1)
    if first_byte in (b"", b"\x00"):      # empty or not-yet-synced file
        audit_rows.append({"file": path.name, "status": "skipped"})
        continue
    records = [raw_to_clean(r, path.name) for r in read_jsonl(path)]
    kept, counts = clean_records(records)
    write_jsonl(CLEAN_DIR / path.name, kept)
    audit_rows.append({"file": path.name, "status": "processed", **counts})

audit = pd.DataFrame(audit_rows)
audit.to_csv(CLEAN_DIR / "cleaning_audit.csv", index=False)
audit

## Step 5. Combine, add emoji columns, restrict dates, sort and assign batches

In [ ]:
import emoji

records = []
for path in sorted(CLEAN_DIR.glob("*.jsonl")):
    records.extend(read_jsonl(path))
print(f"Combined records: {len(records):,}")

# Emoji columns
for r in records:
    found = emoji.emoji_list(str(r.get("combined_text", "")))
    r["is_emoji"] = int(bool(found))
    names, seen = [], set()
    for e in found:
        if e["emoji"] not in seen:
            seen.add(e["emoji"])
            names.append(emoji.demojize(e["emoji"]).strip(":").replace("_", " "))
    r["emoji_name"] = ", ".join(names)

# Duplicate post IDs across files
seen, deduped = set(), []
for r in records:
    pid = r.get("reddit_post_id")
    if pid is None or pid not in seen:
        if pid is not None:
            seen.add(pid)
        deduped.append(r)
records = deduped

# Date window, then newest first
records = [r for r in records if DATE_FROM <= (r.get("created_date") or "") <= DATE_TO]
records.sort(key=lambda r: r.get("created_date", ""), reverse=True)
with open(COMBINED_FILE, "w", encoding="utf-8") as fh:
    for rec in records:
        fh.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Fixed-size batches (batch 1 = most recent posts)
batch_df = pd.DataFrame({
    "record_index": range(len(records)),
    "reddit_post_id": [r.get("reddit_post_id", r.get("id", "")) for r in records],
    "created_date": [r.get("created_date", "") for r in records],
    "batch_number": [i // BATCH_SIZE + 1 for i in range(len(records))],
})
batch_df.to_csv(BATCH_MAP_FILE, index=False)
NUM_BATCHES = math.ceil(len(records) / BATCH_SIZE)
(batch_df.groupby("batch_number")
    .agg(records=("record_index", "count"), date_from=("created_date", "min"), date_to=("created_date", "max")))

## Step 6. LLM extraction (Gemini 2.5 Flash-Lite)

Set `BATCH_TO_RUN` and run the last cell once per batch; it resumes from where it stopped. The key is read from `GEMINI_API_KEY` by the Google GenAI client.

In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from tenacity import retry, stop_after_attempt, wait_random_exponential
from tqdm.asyncio import tqdm as tqdm_asyncio

if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("Set the GEMINI_API_KEY environment variable before running Step 6.")

class ExtractionResult(BaseModel):
    # ── Demographics ─────────────────────────────────────────────────
    age: Optional[str] = Field(description="Age of the author if mentioned (e.g., '25', '30s', 'teenager'), otherwise null")
    gender: Optional[str] = Field(description="Gender or sex of the author if mentioned or implied (e.g., 'male', 'female', 'non-binary', 'M', 'F', '25M', '30F'), otherwise null")
    location: Optional[str] = Field(description="Demographic location mentioned in the text (e.g., London, New York, UK, USA, California), otherwise null")
    # ── Medication & Dosing ──────────────────────────────────────────
    relevant_drugs: Optional[str] = Field(description="All brand names, generic names, or suspected medications mentioned, comma-separated")
    current_dose: Optional[str] = Field(description="Current or most recent dose mentioned by the author (e.g., '2.4mg', '15mg', '5mg tirzepatide', 'highest dose pen', 'maintenance dose'), otherwise null")
    dosing_interval: Optional[str] = Field(description="Dosing interval mentioned by the author (e.g., 'every 7 days', 'every 14 days', 'twice a month', 'monthly', 'weekly'), otherwise null")
    dose_spreading: Optional[str] = Field(description="Whether the author mentions intentionally extending or spreading out the dosing interval beyond the recommended 7 days (e.g., 'stretching to every 2 weeks', 'injecting every 10 days', 'going from weekly to fortnightly') — include reason if given, otherwise null")
    pen_click_counting: Optional[str] = Field(description="Any mention of counting pen clicks to measure, split, or micro-dose from an injection pen in order to stretch a pen further or reduce cost (e.g., 'counted the clicks', 'X clicks = Y mg', 'splitting the pen by clicks', 'used half a pen by counting clicks', 'click method') — include the specific click counts or dose amounts mentioned if given, otherwise null")
    cost_management_strategy: Optional[str] = Field(description="Any strategies mentioned to manage the cost of medication (e.g., pausing treatment, spreading doses, switching to compounded version, splitting pens, click counting, using a lower dose pen, concerns about private prescription cost), otherwise null")
    treatment_interruption: Optional[str] = Field(description="Any mention of stopping, pausing, or taking a break from treatment — include duration and reason if given (e.g., 'stopped for 2 months', 'taking a drug holiday', 'paused because too expensive'), otherwise null")
    restart_pattern: Optional[str] = Field(description="Any mention of restarting medication after a break — include the trigger and any titration details if given (e.g., 'restarted when weight increased', 'went back on after regaining 10 lbs', 'had to re-titrate from the lowest dose'), otherwise null")
    # ── Clinical & Pharmacovigilance ─────────────────────────────────
    indication: Optional[str] = Field(description="The reason the patient was taking the medication (e.g., weight loss, obesity, T2D, PCOS)")
    adverse_events: Optional[str] = Field(description="All symptoms, side effects, or medical events described, comma-separated")
    body_composition: Optional[str] = Field(description="Any mention of muscle loss, fat loss, lean mass, sarcopenia, body composition changes, or concerns about losing muscle vs fat (e.g., 'losing muscle not fat', 'worried about muscle wasting', 'lost skeletal muscle', 'body recomposition'), otherwise null")
    outcome: Optional[str] = Field(description="Seriousness or outcome if mentioned (e.g., hospitalized, recovered, ongoing, discontinued)")
    causality_hints: Optional[str] = Field(description="Any mention of timing (e.g., '3 days after starting') or dechallenge/rechallenge (e.g., 'stopped after I quit the drug')")
    sentiment_perceptions: Optional[str] = Field(description="Emotional tone (e.g., anxious, frustrated, hopeful) and reasons for medication non-compliance or discontinuation if mentioned")
    drug_abuse_misuse: Optional[str] = Field(description="Any patterns of misuse, addiction, overdose, or slang for illicit drug activities, otherwise null")
    off_label_use: Optional[str] = Field(description="Any mention of using the drug outside its approved indication, otherwise null")
    drug_drug_interaction: Optional[str] = Field(description="Any mention of multiple medications taken together that resulted in a reaction or interaction, otherwise null")
    external_stressors: Optional[str] = Field(description="Major life events or environmental factors mentioned (e.g., COVID-19, job loss, bereavement, financial stress, recent surgery, divorce, trauma), comma-separated, otherwise null")

EXTRACTION_FIELDS = [
    'age', 'gender', 'location',
    'relevant_drugs', 'current_dose', 'dosing_interval', 'dose_spreading',
    'pen_click_counting', 'cost_management_strategy',
    'treatment_interruption', 'restart_pattern',
    'indication', 'adverse_events', 'body_composition', 'outcome',
    'causality_hints', 'sentiment_perceptions', 'drug_abuse_misuse',
    'off_label_use', 'drug_drug_interaction', 'external_stressors'
]

EXTRACTION_FIELDS = list(ExtractionResult.model_fields)   # 21 fields, in schema order
assert len(EXTRACTION_FIELDS) == 21

In [ ]:
def build_prompt(text: str) -> str:
    prompt = (
        "You are a pharmacovigilance expert analysing Reddit discussions about weight loss medications "
        "(GLP-1 receptor agonists, tirzepatide, semaglutide, liraglutide, etc.). "
        "Extract the following information from the post. Be thorough and extract ALL relevant details. "
        "If information is not present, return null for that field.\n\n"
        "Pay particular attention to:\n"
        "- DOSING INTERVAL: any mention of how frequently the person injects "
        "(weekly, every 10 days, fortnightly, monthly, etc.).\n"
        "- DOSE SPREADING: any deliberate extension of the injection interval beyond the "
        "recommended 7 days, especially to reduce cost (e.g., 'stretching to every 2 weeks', "
        "'going fortnightly', 'making the pen last longer').\n"
        "- PEN CLICK COUNTING: any mention of counting clicks on the injection pen dial to "
        "measure or split doses (e.g., 'X clicks = Y mg', 'counted the clicks', 'split the pen "
        "by counting clicks', 'click method to micro-dose'). Include specific click counts or "
        "dose amounts if mentioned.\n"
        "- CURRENT DOSE: the dose the person is on, especially if they have reached the "
        "highest available pen (e.g., 15mg tirzepatide, 2.4mg semaglutide).\n"
        "- COST MANAGEMENT: strategies to reduce medication cost — pausing, dose spreading, "
        "click counting to split pens, switching to compounded versions, splitting pens, "
        "or general comments about expense.\n"
        "- TREATMENT INTERRUPTIONS: any stopping or pausing of treatment and for how long "
        "(e.g., 'stopped for 2 months', 'drug holiday', 'took a break').\n"
        "- RESTART PATTERNS: returning to medication after a break and what triggered the restart "
        "(e.g., weight regain, affordability, supply restored).\n"
        "- BODY COMPOSITION: any mention of muscle loss versus fat loss, lean mass, sarcopenia, "
        "or concern about losing skeletal muscle while on these medications.\n\n"
        f"Text:\n'''{text}'''"
    )
    return prompt


client = genai.Client()   # reads GEMINI_API_KEY from the environment


@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(5), reraise=True)
async def extract(record: dict, semaphore: asyncio.Semaphore) -> dict:
    async with semaphore:
        response = await client.aio.models.generate_content(
            model=MODEL,
            contents=build_prompt(str(record.get("combined_text", ""))),
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=ExtractionResult,
                temperature=TEMPERATURE,
            ),
        )
        extracted = response.parsed.model_dump() if response.parsed else {}
    row = dict(record)
    for field in EXTRACTION_FIELDS:
        row[field] = extracted.get(field)
    return row


async def run_batch(batch_number: int) -> None:
    batch_df = pd.read_csv(BATCH_MAP_FILE)
    assert 1 <= batch_number <= int(batch_df["batch_number"].max())
    ids = set(batch_df.loc[batch_df["batch_number"] == batch_number, "reddit_post_id"].astype(str))
    with open(COMBINED_FILE, "r", encoding="utf-8") as fh:
        records = [r for r in map(json.loads, filter(str.strip, fh)) if str(r.get("reddit_post_id", "")) in ids]

    out_file = LLM_DIR / f"gemini_results_batch_{batch_number}.csv"
    done = set()
    columns = list(records[0].keys()) + EXTRACTION_FIELDS
    if out_file.exists():
        done = set(pd.read_csv(out_file, usecols=["reddit_post_id"])["reddit_post_id"].astype(str))
    else:
        with open(out_file, "w", newline="", encoding="utf-8") as fh:
            csv.DictWriter(fh, fieldnames=columns).writeheader()
    todo = [r for r in records if str(r["reddit_post_id"]) not in done]
    print(f"Batch {batch_number}: {len(records):,} posts, {len(done):,} done, {len(todo):,} to run")

    semaphore, lock = asyncio.Semaphore(CONCURRENCY), asyncio.Lock()
    saved = failed = 0

    async def worker(rec):
        nonlocal saved
        row = await extract(rec, semaphore)
        async with lock:
            with open(out_file, "a", newline="", encoding="utf-8") as fh:
                csv.DictWriter(fh, fieldnames=columns).writerow({k: row.get(k) for k in columns})
            saved += 1

    for task in tqdm_asyncio(asyncio.as_completed([worker(r) for r in todo]), total=len(todo)):
        try:
            await task
        except Exception as exc:                      # rate limit or persistent API error
            failed += 1
            if any(k in str(exc) for k in ("RESOURCE_EXHAUSTED", "429", "503", "UNAVAILABLE")):
                print("Rate limit reached; re-run this batch to resume.")
                break
    print(f"Saved {saved:,}; failed {failed:,}; file: {out_file}")

In [ ]:
BATCH_TO_RUN = 1          # change and re-run for each batch (1-20 were run for this study)
await run_batch(BATCH_TO_RUN)

## Step 7. Concatenate batch outputs

In [ ]:
parts = [pd.read_csv(p, dtype=str) for p in sorted(LLM_DIR.glob("gemini_results_batch_*.csv"))]
results = pd.concat(parts, ignore_index=True)
results = results.drop_duplicates(subset="reddit_post_id")
results.to_csv(RESULTS_FILE, index=False)
print(f"{len(parts)} batch files; {len(results):,} posts with extraction results")

## Step 8. Analytic corpus and derived analysis fields

Exclusions: posts whose body was deleted or removed at collection; duplicate posts (identical title and body, earliest kept); posts in which the model's drug field names none of the three target molecules. Posts naming exactly one molecule are assigned to it; posts naming more than one form a separate group.

Derived fields: a model field counts as present when it is non-empty. *Extended interval* is a regular expression on the model's dosing-interval text; the *composite* is positive if any of cost strategy, extended interval, dose spreading, treatment stop or restart is present; *muscle or body-composition concern* is a keyword rule on the body-composition field, the adverse-event field and the post text (Multimedia Appendix 2).

In [ ]:
MOLECULES = {
    "tirzepatide": r"tirzepatide|mounjaro|zepbound",
    "semaglutide": r"semaglutide|ozempic|wegovy",
    "liraglutide": r"liraglutide|saxenda|victoza",
}
EXTENDED_INTERVAL_RE = (r"every other week|every 2 weeks|every 14 days|every 10 days|every 8 days"
                        r"|biweekly|every 4 weeks|monthly")
MUSCLE_TERMS = ["muscle loss", "muscle mass", "lean mass", "lean body", "sarcopenia", "atrophy",
                "skeletal muscle", "loss of muscle", "losing muscle", "muscle wasting", "muscle",
                "body composition", "lean tissue"]


def assign_molecule(drug_field) -> Optional[str]:
    if pd.isna(drug_field):
        return None
    text = " ".join(p.strip().lower() for p in re.split(r"[;,\n]+", str(drug_field)) if p.strip())
    hits = [m for m, pat in MOLECULES.items() if re.search(pat, text)]
    if not hits:
        return None
    return hits[0] if len(hits) == 1 else "multiple"


def build_analytic_corpus(results_csv: Path) -> pd.DataFrame:
    cols = ["reddit_post_id", "created_date", "author", "subreddit", "relevant_drugs", "dosing_interval",
            "dose_spreading", "treatment_interruption", "restart_pattern", "cost_management_strategy",
            "body_composition", "adverse_events", "combined_text", "selftext", "title"]
    df = pd.read_csv(results_csv, usecols=cols, dtype=str)
    df["post_dt"] = pd.to_datetime(df["created_date"], errors="coerce")
    df = df.dropna(subset=["post_dt"])
    df = df[(df["post_dt"] >= ANALYSIS_FROM) & (df["post_dt"] <= ANALYSIS_TO)]
    flow = {"posts in extraction output": len(df)}

    removed = df["selftext"].isin(["[removed]", "[deleted]"])
    df = df[~removed].sort_values("post_dt")
    flow["excluded: body deleted or removed"] = int(removed.sum())

    key = df["title"].fillna("") + "\u241f" + df["selftext"].fillna("")
    dup = key.duplicated(keep="first") & df["selftext"].fillna("").str.len().gt(0)
    df = df[~dup].copy()
    flow["excluded: duplicate title and body"] = int(dup.sum())

    df["stratum"] = df["relevant_drugs"].apply(assign_molecule)
    flow["excluded: no target molecule"] = int(df["stratum"].isna().sum())
    df = df[df["stratum"].notna()].copy()
    flow["analytic corpus"] = len(df)

    present = lambda col: df[col].fillna("").astype(str).str.strip().ne("")
    df["cost_strategy_flag"] = present("cost_management_strategy")
    df["extended_interval"] = df["dosing_interval"].fillna("").str.lower().str.contains(EXTENDED_INTERVAL_RE, na=False)
    df["dose_spreading_flag"] = present("dose_spreading")
    df["treatment_stop_flag"] = present("treatment_interruption")
    df["restart_flag"] = present("restart_pattern")
    df["composite_flag"] = (df["cost_strategy_flag"] | df["extended_interval"] | df["dose_spreading_flag"]
                            | df["treatment_stop_flag"] | df["restart_flag"])
    text = (df["body_composition"].fillna("").str.lower() + " " + df["adverse_events"].fillna("").str.lower()
            + " " + df["combined_text"].fillna("").str.lower())
    df["muscle_concern_flag"] = text.apply(lambda t: any(term in t for term in MUSCLE_TERMS))
    print(pd.Series(flow).to_string())
    return df


corpus = build_analytic_corpus(RESULTS_FILE)
corpus.drop(columns=["selftext", "title", "combined_text"]).to_csv(DATA_DIR / "analytic_corpus.csv", index=False)

flags = ["composite_flag", "treatment_stop_flag", "restart_flag", "dose_spreading_flag", "muscle_concern_flag"]
summary = corpus.groupby("stratum")[flags].mean().mul(100).round(1)
summary.loc["all posts"] = corpus[flags].mean().mul(100).round(1)
summary